# 04 — Model Training and Evaluation

This notebook focuses on developing, training, and evaluating machine learning models for dry-spell warning prediction.

The workflow includes:

- Loading the prepared datasets.
- Establishing a baseline model.
- Evaluating model performance using appropriate classification metrics.
- Comparing models and tuning decision thresholds.
- Selecting a final model based on validation performance.

All experiments will use chronological validation to reflect prediction on future observations.

## 4.1 Environment Setup and Data Loading

This section imports the required libraries and loads the prepared datasets and feature engineering functions.

The data will be reconstructed from the original source files using the project modules to ensure that this notebook can run independently of other notebooks.

### 4.1.1 Imports and Load Data

The original training and testing datasets are loaded using the project's reusable data loading functions.

The datasets are merged and standardized before feature engineering.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Set project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Project configuration
from config import TRAIN_DIR, TEST_DIR

# Source modules
from src.data_loader import (
    load_and_standardize_datasets,
    merge_datasets
)

from src.feature_engineering import (
    add_lag_and_rolling_features,
    add_calendar_features
)

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


# Load and merge datasets
train_datasets = load_and_standardize_datasets(TRAIN_DIR)
test_datasets = load_and_standardize_datasets(TEST_DIR)

train_merged = merge_datasets(train_datasets)
test_merged = merge_datasets(test_datasets)

print("Training data:", train_merged.shape)
print("Testing data:", test_merged.shape)

Training data: (2214, 18)
Testing data: (738, 17)


### 4.1.3 Load Prepared Modeling Data

The prepared modeling datasets are loaded from the processed data directory.

The training and validation sets will be used for model development and evaluation. The test features will remain separate until the final evaluation stage.

In [2]:
processed_path = Path("../data/processed/modeling_data.pkl")

prepared_data = pd.read_pickle(processed_path)

X_train = prepared_data["X_train"]
y_train = prepared_data["y_train"]

X_val = prepared_data["X_val"]
y_val = prepared_data["y_val"]

X_test = prepared_data["X_test"]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape)

Training: (1722, 64) (1722,)
Validation: (492, 64) (492,)
Testing: (738, 64)


## 4.2 Baseline Model

A baseline model establishes a simple reference point for evaluating more advanced machine learning models.

A Dummy Classifier will be trained using the most frequent class strategy. Its performance will help demonstrate why accuracy alone is insufficient for this imbalanced classification problem.

In [3]:
baseline_model = DummyClassifier(strategy="most_frequent")

baseline_model.fit(X_train, y_train)

y_val_pred_baseline = baseline_model.predict(X_val)

print("Baseline Model — Validation Results")
print("-----------------------------------")
print(f"Accuracy:  {accuracy_score(y_val, y_val_pred_baseline):.4f}")
print(f"Precision: {precision_score(y_val, y_val_pred_baseline, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_val, y_val_pred_baseline, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_val, y_val_pred_baseline, zero_division=0):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_baseline))

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_baseline, zero_division=0))

Baseline Model — Validation Results
-----------------------------------
Accuracy:  0.9248
Precision: 0.0000
Recall:    0.0000
F1-score:  0.0000

Confusion Matrix:
[[455   0]
 [ 37   0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96       455
           1       0.00      0.00      0.00        37

    accuracy                           0.92       492
   macro avg       0.46      0.50      0.48       492
weighted avg       0.86      0.92      0.89       492



### Key Findings

- The baseline model achieved 92.48% validation accuracy by predicting every sample as the majority class (Class 0).
- The model failed to identify any positive dry spell warning cases.
- Class 1 precision, recall, and F1-score are all 0.00.
- The confusion matrix shows 37 false negatives, meaning all positive validation cases were missed.
- Accuracy is misleading for this imbalanced dataset and should not be used as the primary evaluation metric.

The baseline confirms that a useful model must learn to detect the minority class rather than simply predict the majority class.

## 4.3 Logistic Regression

Logistic Regression is used as the first machine learning model to learn patterns associated with dry spell warning cases.

The pipeline includes median imputation for missing values, feature scaling, and class weighting to address the imbalanced target distribution.

The model will be trained on the training set and evaluated on the validation set.


In [4]:
logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

y_val_pred_lr = logistic_model.predict(X_val)

print("Logistic Regression — Validation Results")
print("---------------------------------------")
print(f"Accuracy:  {accuracy_score(y_val, y_val_pred_lr):.4f}")
print(f"Precision: {precision_score(y_val, y_val_pred_lr, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_val, y_val_pred_lr, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_val, y_val_pred_lr, zero_division=0):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_lr))

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_lr, zero_division=0))

Logistic Regression — Validation Results
---------------------------------------
Accuracy:  0.8659
Precision: 0.2364
Recall:    0.3514
F1-score:  0.2826

Confusion Matrix:
[[413  42]
 [ 24  13]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.91      0.93       455
           1       0.24      0.35      0.28        37

    accuracy                           0.87       492
   macro avg       0.59      0.63      0.60       492
weighted avg       0.89      0.87      0.88       492



### Key Findings

- Logistic Regression achieved 86.59% validation accuracy.
- The model identified 13 out of 37 positive dry spell warning cases, achieving a recall of 35.14%.
- The model missed 24 positive cases, resulting in 24 false negatives.
- The model produced 42 false positives, leading to a low precision of 23.64%.
- The F1-score for Class 1 was 28.26%, indicating limited balance between precision and recall.
- Compared with the baseline, Logistic Regression successfully detected minority-class cases, but its false alarm rate remains high.

The results show that class weighting enabled minority-class detection, although further model and threshold evaluation is required.

## 4.4 Classification Threshold Analysis

Logistic Regression uses a default classification threshold of 0.5 to convert predicted probabilities into class labels.

In this section, different thresholds are evaluated on the validation set to understand the trade-off between precision and recall for the positive class.

The test set remains untouched.

In [5]:
y_val_proba_lr = logistic_model.predict_proba(X_val)[:, 1]

thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

threshold_results = []

for threshold in thresholds:
    y_pred = (y_val_proba_lr >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_val, y_pred, zero_division=0),
        "Recall": recall_score(y_val, y_pred, zero_division=0),
        "F1-score": f1_score(y_val, y_pred, zero_division=0),
        "Predicted Positives": int(y_pred.sum())
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df

,Threshold,Precision,Recall,F1-score,Predicted Positives
0,0.2,0.219298,0.675676,0.331126,114
1,0.3,0.244444,0.594595,0.346457,90
2,0.4,0.253731,0.459459,0.326923,67
3,0.5,0.236364,0.351351,0.282609,55
4,0.6,0.195652,0.243243,0.216867,46
5,0.7,0.189189,0.189189,0.189189,37


### Key Findings

- Lowering the classification threshold increased recall for the positive class.
- The highest recall was 67.57% at a threshold of 0.2, detecting 25 out of 37 positive cases.
- The highest F1-score was 34.65% at a threshold of 0.3.
- At threshold 0.3, the model detected 22 out of 37 positive cases while predicting 90 positive cases.
- Lower thresholds increased the number of predicted positive cases and false alarms.
- Threshold 0.3 provides the best F1-score among the evaluated thresholds, while threshold 0.2 prioritizes recall.

The threshold should be selected based on the intended balance between detecting dry spell warnings and limiting false alarms.

## 4.5 Random Forest

In this section, we train a Random Forest classifier to compare its performance with the baseline and Logistic Regression models.

Random Forest can capture nonlinear relationships and interactions between features. We use class weighting to address the target imbalance.

The model is evaluated on the validation set only. The test set remains untouched.

In [6]:
random_forest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])

random_forest_model.fit(X_train, y_train)

y_val_pred_rf = random_forest_model.predict(X_val)

print("Random Forest - Validation Results")
print(f"Accuracy:  {accuracy_score(y_val, y_val_pred_rf):.4f}")
print(f"Precision: {precision_score(y_val, y_val_pred_rf, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_val, y_val_pred_rf, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_val, y_val_pred_rf, zero_division=0):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_rf))

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_rf, zero_division=0))

Random Forest - Validation Results
Accuracy:  0.9248
Precision: 0.0000
Recall:    0.0000
F1-score:  0.0000

Confusion Matrix:
[[455   0]
 [ 37   0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96       455
           1       0.00      0.00      0.00        37

    accuracy                           0.92       492
   macro avg       0.46      0.50      0.48       492
weighted avg       0.86      0.92      0.89       492



### Key Findings

- Random Forest achieved 92.48% validation accuracy, but this was caused by predicting every sample as the majority class (Class 0).
- The model failed to detect any of the 37 positive dry spell warning cases.
- Precision, recall, and F1-score for the positive class were all 0.00.
- The confusion matrix shows 37 false negatives and no true positives.
- Random Forest did not outperform the majority-class baseline in detecting dry spell warnings.
- The model requires threshold adjustment and further evaluation before it can be considered useful for minority-class detection.

The results indicate that the default classification threshold of 0.5 is unsuitable for detecting the minority class with this Random Forest model.

## 4.6 Random Forest Threshold Analysis

The default classification threshold of 0.5 may not be suitable for an imbalanced dataset.

In this section, we analyze the predicted probabilities from Random Forest and evaluate alternative thresholds using the validation set.

The goal is to determine whether lowering the threshold can improve dry spell detection. The test set remains untouched.

In [7]:
y_val_proba_rf = random_forest_model.predict_proba(X_val)[:, 1]

print("Probability range:")
print(f"Minimum: {y_val_proba_rf.min():.4f}")
print(f"Maximum: {y_val_proba_rf.max():.4f}")
print(f"Mean:    {y_val_proba_rf.mean():.4f}")

thresholds = [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]

rf_threshold_results = []

for threshold in thresholds:
    y_pred = (y_val_proba_rf >= threshold).astype(int)

    rf_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_val, y_pred, zero_division=0),
        "Recall": recall_score(y_val, y_pred, zero_division=0),
        "F1-score": f1_score(y_val, y_pred, zero_division=0),
        "Predicted Positives": int(y_pred.sum())
    })

rf_threshold_results_df = pd.DataFrame(rf_threshold_results)
rf_threshold_results_df

Probability range:
Minimum: 0.0000
Maximum: 0.4945
Mean:    0.0968


,Threshold,Precision,Recall,F1-score,Predicted Positives
0,0.05,0.207865,1.000000,0.344186,178
1,0.10,0.255172,1.000000,0.406593,145
2,0.15,0.268116,1.000000,0.422857,138
3,0.20,0.302521,0.972973,0.461538,119
4,0.30,0.278481,0.594595,0.379310,79
5,0.40,0.272727,0.081081,0.125000,11
6,0.50,0.000000,0.000000,0.000000,0


### Key Findings

- Random Forest predicted probabilities ranged from 0.0000 to 0.4945, with a mean of 0.0968.
- The default threshold of 0.50 classified all validation samples as negative.
- Lowering the threshold to 0.20 achieved 97.30% recall, detecting 36 out of 37 positive cases.
- At threshold 0.20, precision was 30.25% and the F1-score was 46.15%.
- The model predicted 119 positive cases, including 83 false positives.
- Threshold 0.20 achieved the highest F1-score among the tested thresholds.

The results show that lowering the classification threshold substantially improved dry spell detection, although false alarms remain relatively high.